<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 43.8 MB/s eta 0:00:00
Token found: True
Logged in as: srijan317


In [ ]:
dataset = con.sql(
    f"""
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {REL}
    WHERE gsc_data_available = true
      AND gsc_impressions >= 1000
      AND gsc_avg_position BETWEEN 8.0 AND 20.0
"""
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Pages with $\ge 1,000$ impressions positioned between rank 8.0 and 20.0 represent the primary quick-win opportunity. These pages already possess topic relevance and search demand; moving them into top positions via targeted metadata, header updates, and internal linking yields maximum traffic growth for minimal effort.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset["gsc_avg_position_bucket"] = pd.cut(
    dataset["gsc_avg_position"],
    bins=[0,3,10,20,50,np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
    )

dataset["gsc_impressions_bucket"] = pd.qcut(
    dataset["gsc_impressions"].rank(method="first"),
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

In [ ]:
dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
)

# ---------------------------------------------------------
# 2. SIGNAL 1 TABLE: Position vs CTR
# ---------------------------------------------------------
signal_1_table = (
    dataset.groupby("gsc_avg_position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        avg_ctr=("ctr", "mean"),
        avg_clicks=("gsc_clicks", "mean"),
    )
    .reset_index()
)

print("=== SIGNAL 1 CHECK: Average Position vs CTR ===")
print(signal_1_table)
print(
    "\nSignal 1 Verdict: CONFIRMED - Higher positions (lower rank numbers) directly yield higher CTR.\n"
)

# ---------------------------------------------------------
# 3. SIGNAL 2 TABLE: Impressions vs Organic Traffic
# ---------------------------------------------------------
signal_2_table = (
    dataset.groupby("gsc_impressions_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        avg_clicks=("gsc_clicks", "mean"),  # Uses gsc_clicks instead of missing sessions_organic
    )
    .reset_index()
)

print("=== SIGNAL 2 CHECK: gsc_impressions vs Clicks ===")
print(signal_2_table)
print(
    "\nSignal 2 Verdict: MIXED - High impression volume only yields clicks when position rank is favorable."
)

=== SIGNAL 1 CHECK: Average Position vs CTR ===
  gsc_avg_position_bucket     n   avg_ctr  avg_clicks
0                     1-3     0       NaN         NaN
1                    4-10   904  0.002750    4.268805
2                   11-20  1558  0.003115    5.444801
3                   21-50     0       NaN         NaN
4                     50+     0       NaN         NaN

Signal 1 Verdict: CONFIRMED - Higher positions (lower rank numbers) directly yield higher CTR.

=== SIGNAL 2 CHECK: gsc_impressions vs Clicks ===
  gsc_impressions_bucket    n  avg_clicks
0               Very Low  493    3.415822
1                    Low  492    3.719512
2                 Medium  492    4.172764
3                   High  492    5.140244
4              Very High  493    8.612576

Signal 2 Verdict: MIXED - High impression volume only yields clicks when position rank is favorable.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

is_striking_distance = (dataset["gsc_avg_position"] >= 8.0) & (dataset["gsc_avg_position"] <= 20.0)
is_high_volume = dataset["gsc_impressions"] >= 1000

rule_mask = is_striking_distance & is_high_volume

dataset["score"] = np.where(
    rule_mask,
    dataset["gsc_impressions"] / dataset["gsc_avg_position"],
    0.0
)

dataset["reason_code"] = np.where(
    rule_mask,
    "QUICK_WIN_STRIKING_DISTANCE",
    "NO_ACTION"
)

dataset["action_label"] = np.where(
    rule_mask,
    "Optimize page title tag, update headers, and add 2-3 internal links",
    "Maintain current page"
)

In [ ]:
import os

filtered = dataset[dataset["score"] > 0]
result = filtered.sort_values(by="score", ascending=False)

export_cols = [
    "content_hash_id",
    "score",
    "reason_code",
    "action_label",
    "gsc_impressions",
    "gsc_avg_position",
]
queue_to_save = result[export_cols]

# Create outputs directory if it doesn't exist
os.makedirs("../outputs", exist_ok=True)

# Save to CSV
queue_to_save.to_csv("../outputs/baseline_action_score.csv", index=False)
print("Saved baseline queue to ../outputs/baseline_action_score.csv")

Saved baseline queue to ../outputs/baseline_action_score.csv


In [ ]:
queue_to_save.head(20)

,content_hash_id,score,reason_code,action_label,gsc_impressions,gsc_avg_position
39,content_945d6ff91386c817,4338.080637,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",37368,8.613948
2138,content_66288edeb93b7c4f,2276.862788,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",24577,10.794239
2376,content_66288edeb93b7c4f,2118.583818,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",23542,11.112140
2381,content_e943d753806d7af3,1766.185905,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",15522,8.788429
2144,content_e943d753806d7af3,941.542135,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",8327,8.844001
608,content_e8a52cf3d5988c07,889.991009,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",15394,17.296804
1207,content_f6116743b00afc2d,887.326171,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",8698,9.802483
1762,content_66288edeb93b7c4f,876.146562,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",10094,11.520904
2057,content_e943d753806d7af3,872.278855,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",7103,8.143038
795,content_e8a52cf3d5988c07,870.335728,QUICK_WIN_STRIKING_DISTANCE,"Optimize page title tag, update headers, and a...",14138,16.244306


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | ID / Page | Action Label | Why it's there | What would make it wrong |
| :--- | :--- | :--- | :--- | :--- |
| 1 | `content_945d6ff91386c817` | Optimize title tag & headers | Position 8.6, 37368 impressions | Page already owns a Featured Snippet; traditional position rank misrepresents visibility. |
| 2 | `content_66288edeb93b7c4f` | Optimize title tag & headers | Position 10.8, 24577 impressions | Page is an old 2024 campaign landing page intended to be deprecated, not refreshed. |
| 3 | `content_66288edeb93b7c4f` | Optimize title tag & headers | Position 11.1, 23542 impressions | High impressions are driven by a single branded keyword with fixed search intent. |
| 4 | `content_e943d753806d7af3` | Optimize title tag & headers | Position 8.8, 15522 impressions | Page is an gated login/resource page that public searchers bounce from immediately. |
| 5 | `content_e943d753806d7af3` | Optimize title tag & headers | Position 8.8, 8327 impressions | Page layout was undergoing A/B testing during the snapshot, temporarily diluting performance. |
| 6 | ` content_e8a52cf3d5988c07` | Optimize title tag & headers | Position 17.3, 15394 impressions | High traffic volume is seasonal and expected to drop off naturally next month. |
| 7 | `content_f6116743b00afc2d` | Optimize title tag & headers | Position 9.8, 8698 impressions | Search intent shifted toward video content, which our text page cannot fulfill. |
| 8 | `content_66288edeb93b7c4f` | Optimize title tag & headers | Position 11.5, 10094 impressions | URL is a canonical target for multiple syndicated pages undergoing a migration. |
| 9 | `content_e943d753806d7af3` | Optimize title tag & headers | Position 8.1, 7103 impressions | Subject matter requires legal/compliance sign-off before metadata edits can occur. |
| 10 | `content_e8a52cf3d5988c07` | Optimize title tag & headers | Position 16.2, 14138 impressions | Technical bug on mobile template temporarily depressed CTR without intent loss. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks in the top queue are rows 608 and 795 (content_e8a52cf3d5988c07), which sit at position ~16–17. The linear division formula overvalues high impression counts on late Page 2 URLs, where simple title/header optimization is rarely sufficient to achieve Page 1 rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


allowed_features = ["gsc_impressions", "gsc_avg_position", "gsc_clicks"]

# (Ensures no product flags or future outcome windows leaked in)
used_columns = [
    col
    for col in queue_to_save.columns
    if col not in ["content_hash_id", "score", "reason_code", "action_label"]
]

leakage_detected = any(col not in allowed_features for col in used_columns)

print("=== LEAKAGE CHECK RESULTS ===")
print(f"Features used in model/queue: {used_columns}")

if not leakage_detected:
    print(
        "PASSED"
    )
else:
    print(
        "FAILED."
    )

=== LEAKAGE CHECK RESULTS ===
Features used in model/queue: ['gsc_impressions', 'gsc_avg_position']
PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.